<a href="https://colab.research.google.com/github/Tahsin22201243/Machine-Learning/blob/main/Mega3_22201243.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import os
import random
import re
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import (
    Dataset,
    DataLoader,
    TensorDataset,
    WeightedRandomSampler
)

from torchvision import transforms
import torchvision.transforms.functional as Ff

from sklearn.datasets import (
    load_digits,
    fetch_20newsgroups
)

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)


In [8]:
import numpy as np
import torch
import  torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.datasets import fetch_lfw_people

**Part A**
Qns 1

In [9]:
#Qns 1
from sklearn.preprocessing import MinMaxScaler
lfw = fetch_lfw_people(min_faces_per_person=70,resize=0.4)
x= lfw.data
y= lfw.target
print("Orginal Shape:",x.shape)
print("Classes:",np.unique(y))


Orginal Shape: (1288, 1850)
Classes: [0 1 2 3 4 5 6]


In [10]:
#Qns 2
scaler = MinMaxScaler()
X = scaler.fit_transform(X)


In [17]:
import pandas as pd
import numpy as np

In [18]:
#Qns 3
class_1_idx = np.where(y == 1)[0]
other_idx = np.where(y != 1)[0]

np.random.seed(42)

keep_class1 = np.random.choice(
    class_1_idx,
    size=int(0.25 * len(class_1_idx)),
    replace=False
)

final_idx = np.concatenate([keep_class1, other_idx])

X = X[final_idx]
y = y[final_idx]

print("After Imbalance:", X.shape)

#Qns 4

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)
#Qns 5

X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val = torch.tensor(y_val, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

#Qns 6
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

#Qns 7

class_counts = np.bincount(y_train.numpy())

weights = 1. / class_counts

sample_weights = weights[y_train.numpy()]

sample_weights = torch.DoubleTensor(sample_weights)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

#Qns 8
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    sampler=sampler
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False
)

#Qns 9

input_size = X_train.shape[1]
num_classes = len(torch.unique(y_train))

class DNN(nn.Module):

    def __init__(self):
        super(DNN, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.ReLU(),

            nn.Linear(512, 128),
            nn.ReLU(),

            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.model(x)

model = DNN()

print(model)

#Qns 10

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)



def accuracy(loader):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for X_batch, y_batch in loader:

            outputs = model(X_batch)

            _, predicted = torch.max(outputs, 1)

            total += y_batch.size(0)

            correct += (predicted == y_batch).sum().item()

    return 100 * correct / total

#Qns 11
epochs = 5

for epoch in range(epochs):

    model.train()

    running_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    train_acc = accuracy(train_loader)

    val_acc = accuracy(val_loader)

    print(f"Epoch [{epoch+1}/{epochs}]")
    print(f"Loss: {running_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.2f}%")
    print(f"Validation Accuracy: {val_acc:.2f}%")
    print("-" * 40)

#Qns 12

test_acc = accuracy(test_loader)

print(f"Final Test Accuracy: {test_acc:.2f}%")

IndexError: index 361 is out of bounds for axis 0 with size 178

**Part B Qns 2**

In [ ]:

import torch
import torchvision
import matplotlib.pyplot as plt

from torchvision import transforms
from torch.utils.data import DataLoader

#Qns1

train_transform = transforms.Compose([

    transforms.RandomHorizontalFlip(),

    transforms.RandomRotation(15),

    transforms.ToTensor()
])

#Qns2

train_dataset = torchvision.datasets.STL10(
    root='./data',
    split='train',
    download=True,
    transform=train_transform
)

#Qns3
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

#Qns4

images, labels = next(iter(train_loader))

print("Image Shape:", images.shape)

#Qns5

fig, axes = plt.subplots(2, 4, figsize=(10, 5))

for i in range(8):

    img = images[i].permute(1, 2, 0)

    ax = axes[i // 4, i % 4]

    ax.imshow(img)

    ax.axis("off")

plt.tight_layout()

plt.show()

**Part C**

In [39]:


import pandas as pd
import numpy as np
import re

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
#Qns 1

df = pd.read_csv("/content/tweets.csv")

print(df.head(10))
print("Available columns:", df.columns.tolist())

texts = df['content'].values
labels = df['TargetEncoder'].values

#Qns 2

def preprocess(text):

    text = text.lower()

    text = re.sub(r'[^a-zA-Z ]', '', text)

    tokens = text.split()

    return tokens

tokenized_texts = [preprocess(text) for text in texts]


vocab = {}

idx = 1

for tokens in tokenized_texts:

    for word in tokens:

        if word not in vocab:

            vocab[word] = idx

            idx += 1

print("Vocabulary Size :", len(vocab))
#Qns 3

sequences = []

for tokens in tokenized_texts:

    seq = []

    for word in tokens:

        seq.append(vocab[word])

    sequences.append(seq)

#Qns 4

max_len = 60

padded_sequences = []

for seq in sequences:

    if len(seq) < max_len:

        seq = seq + [0] * (max_len - len(seq))

    else:

        seq = seq[:max_len]

    padded_sequences.append(seq)

X = np.array(padded_sequences)

y = np.array(labels)

print("Input Shape :", X.shape)


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


X_train = torch.tensor(X_train, dtype=torch.long)
X_test = torch.tensor(X_test, dtype=torch.long)

y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)



train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

#Qns 5


class BiLSTM(nn.Module):

    def __init__(self, vocab_size, embed_dim, hidden_dim):

        super(BiLSTM, self).__init__()


        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=0
        )


        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )


        self.fc = nn.Linear(hidden_dim * 2, 2)

    def forward(self, x):

        x = self.embedding(x)

        output, (hidden, cell) = self.lstm(x)


        forward_hidden = hidden[-2]
        backward_hidden = hidden[-1]

        hidden_concat = torch.cat(
            (forward_hidden, backward_hidden),
            dim=1
        )

        out = self.fc(hidden_concat)

        return out



vocab_size = len(vocab) + 1

model = BiLSTM(
    vocab_size=vocab_size,
    embed_dim=128,
    hidden_dim=64
)

print(model)



criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

#Qns 6

epochs = 3

for epoch in range(epochs):

    model.train()

    running_loss = 0

    correct = 0
    total = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += y_batch.size(0)

        correct += (predicted == y_batch).sum().item()

    accuracy = 100 * correct / total

    # #Qns 7


    print(f"\nEpoch [{epoch+1}/{epochs}]")

    print("Training Loss :", running_loss)

    print("Training Accuracy :", accuracy)

#Qns 8

def predict_tweet(tweet):


    tokens = preprocess(tweet)

    seq = []

    for word in tokens:

        seq.append(vocab.get(word, 0))


    if len(seq) < max_len:

        seq = seq + [0] * (max_len - len(seq))

    else:

        seq = seq[:max_len]

    seq = torch.tensor([seq], dtype=torch.long)

    model.eval()

    with torch.no_grad():

        output = model(seq)

        prediction = torch.argmax(output, dim=1).item()

    if prediction == 1:

        print("Disaster Tweet")

    else:

        print("Non-Disaster Tweet")

predict_tweet("Massive earthquake destroyed the city")

predict_tweet("I am watching cricket with friends")


      author                                            content country  \
0  katyperry  Is history repeating itself...?#DONTNORMALIZEH...     NaN   
1  katyperry  @barackobama Thank you for your incredible gra...     NaN   
2  katyperry                Life goals. https://t.co/XIn1qKMKQl     NaN   
3  katyperry            Me right now 🙏🏻 https://t.co/gW55C1wrwd     NaN   
4  katyperry  SISTERS ARE DOIN' IT FOR THEMSELVES! 🙌🏻💪🏻❤️ ht...     NaN   
5  katyperry  happy 96th gma #fourmoreyears! 🎈 @ LACMA Los A...     NaN   
6  katyperry  Kyoto, Japan \r\n1. 5. 17. https://t.co/o28M0v...     NaN   
7  katyperry       🇯🇵 @ Sanrio Puroland https://t.co/eXVev5UMBx     NaN   
8  katyperry           2017 resolution: to embody authenticity!     NaN   
9  katyperry                   sisters. https://t.co/5ZE21x2aNk     NaN   

          date_time            id language  latitude  longitude  \
0  12/01/2017 19:52  8.196330e+17       en       NaN        NaN   
1  11/01/2017 08:38  8.191010e+17       

KeyError: 'TargetEncoder'

**Part D**

In [22]:
#Qns 1


import torch
from torch.utils.tensorboard import SummaryWriter



writer = SummaryWriter("./runs/")

#Qns 2

epochs = 2

for epoch in range(epochs):

    model.train()

    running_loss = 0

    correct = 0
    total = 0

    for images, labels in train_loader:

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()


    model.eval()

    with torch.no_grad():

        for val_images, val_labels in val_loader:

            outputs = model(val_images)

            _, predicted = torch.max(outputs, 1)

            total += val_labels.size(0)

            correct += (predicted == val_labels).sum().item()

    val_acc = 100 * correct / total



    writer.add_scalar(
        "Training Loss",
        running_loss,
        epoch
    )

    writer.add_scalar(
        "Validation Accuracy",
        val_acc,
        epoch
    )



    writer.add_images(
        "Sample Images",
        images[:16],
        epoch
    )

    print(f"Epoch {epoch+1}")
    print("Training Loss :", running_loss)
    print("Validation Accuracy :", val_acc)



writer.close()

print("Logs saved in ./runs/")

RuntimeError: mat1 and mat2 shapes cannot be multiplied (2304x96 and 13x128)